In [16]:
import os
import re

P = [0.25, 0.5, 0.75, 0.9]

In [17]:
def parse_dzn_file(file_path):
    with open(file_path, "r") as f:
        content = f.readlines()

    print(f"content:\n{content}")

    # Helper function to extract single integer variables
    def get_int(var_name):
        for line in content:
            match = re.search(fr"^{var_name}\s*=\s*(\d+);", line.strip())
            if match:
                return int(match.group(1))
        return None

    # Helper function to extract arrays of integers
    def get_array(var_name):
        for line in content:
            match = re.search(fr"^{var_name}\s*=\s*\[(.*?)\];", line.strip(), re.DOTALL)
            if match:
                # Clean whitespaces/newlines and split by commas
                clean_str = re.sub(r"\s+", "", match.group(1))
                return [int(x) for x in clean_str.split(",") if x]
        return []
    # Parse structural properties
    num_items = get_int("num_items")
    num_bids = get_int("num_bids")
    total_bid = get_int("totalbid")

    index_packages = get_array("index_packages")
    packages = get_array("packages")
    bids = get_array("bids")

    # Slice the flattened packages array into discrete bid bundles
    # MiniZinc is 1-indexed, so we subtract 1 for Python offsets
    bid_packages = {}
    for i in range(num_bids):
        start_idx = index_packages[i] - 1
        if i == num_bids - 1:
            end_idx = len(packages)
        else:
            end_idx = index_packages[i + 1] - 1

        # Save 1-indexed bid mapping to items
        bid_packages[i + 1] = packages[start_idx:end_idx]

    return {
        "num_bids": num_bids,
        "num_items": num_items,
        "bids": bids,
        "bid_packages": bid_packages,
        "total_bid": total_bid,
    }

In [18]:
parsed = parse_dzn_file("/Users/instafiore/Workspace/AMOSUM/tests/benchmarks/combinatoria_auctions/instances_dly/cat_paths_60_70_0000.dly")

content:
['num_items=86;\n', 'num_bids=70;\n', 'max_item=86;\n', 'index_packages=[1,5,10,14,18,23,24,27,31,32,34,37,41,44,49,54,60,63,67,72,76,81,83,89,96,101,107,109,115,118,122,127,133,134,138,143,146,151,157,164,166,167,170,173,177,184,191,193,197,202,209,212,220,227,230,232,236,240,246,253,255,259,264,270,271,274,278,281,285,287,291];\n', 'tn_packages=290;\n', 'packages=[3,18,49,61,2,4,18,49,61,19,22,42,61,3,19,45,61,3,19,22,30,61,28,13,30,62,4,21,30,62,33,22,27,9,10,26,4,35,50,63,12,58,63,8,13,46,50,63,5,8,23,55,63,5,8,15,23,43,63,43,60,64,20,34,55,64,15,20,34,43,64,11,33,55,64,11,15,33,43,64,36,65,13,17,20,29,51,65,13,17,20,26,29,39,65,13,20,34,46,65,13,25,26,29,38,65,59,66,5,13,29,37,50,66,33,55,67,15,33,43,67,7,14,33,43,67,7,9,32,33,43,67,28,22,32,43,68,9,14,22,43,68,24,54,69,21,24,29,37,69,4,13,24,29,37,69,2,3,13,24,29,37,69,26,29,23,16,44,70,23,53,70,16,23,40,70,5,13,26,29,38,50,70,5,13,16,29,50,51,70,24,56,21,28,29,71,4,13,28,29,71,5,13,26,29,50,52,72,1,59,72,4,5,21,26,29,50

In [19]:
def compute_greedy_groups(data):
    bid_packages = data["bid_packages"]
    num_bids = data["num_bids"]

    # Helper to check if two bids share an item (meaning they are incompatible)
    def are_incompatible(b1, b2):
        return len(set(bid_packages[b1]) & set(bid_packages[b2])) > 0

    # Groups holds a list of sets: [set([bid1, bid2]), set([bid3])]
    groups = []

    for b in range(1, num_bids + 1):
        placed = False
        for g_idx, current_group in enumerate(groups):
            # Check if 'b' is pairwise incompatible with ALL existing bids in this group
            if all(are_incompatible(b, existing) for existing in current_group):
                current_group.add(b)
                placed = True
                break

        # If no such group was compatible to enforce the complete clique property, spawn a new group
        if not placed:
            groups.append({b})

    # Convert groups structure to a flat map: bid -> group_id (1-indexed)
    bid_to_group = {}
    mps = 0
    for g_idx, current_group in enumerate(groups):
        max_w = max([data["bids"][b-1] for b in current_group])
        mps += max_w
        for b in current_group:
            bid_to_group[b] = g_idx + 1

    return bid_to_group, [int(mps*p) for p in P]

In [20]:
def create_rule(head, body = None):
    head = [head] if type(head) != list else head
    rule = ([f"{'|'.join(head)}"] if head else [""]) + ([f"{','.join(body)}"] if body else [])
    return ":- ".join(rule) + "."


In [21]:
def write_plain_asp_file(data, instance_name, dir_name, mode = "w"):
    lines = []

    for b_id, val in enumerate(data["bids"], start=1):
        lines.append(create_rule(f"bid({b_id},{val})"))

    for b_id, items in data["bid_packages"].items():
        for item in items:
            lines.append(create_rule(f"package({b_id},{item})"))

    with open(f"{dir_name}/{instance_name}.asp", mode) as f:
        f.write("\n".join(lines))


In [22]:
def write_amosum_asp_file(data, instance_name, dir_name, mode = "w", best_bounds: dict[str, int] = {}):    
    bid_to_group, bounds = compute_greedy_groups(data)


    lines = []
    for b_id, g_id in bid_to_group.items():
        lines.append(create_rule(f"group({b_id},{g_id})"))

    bounds_to_insert = bounds if best_bounds.get(instance_name, None) is None else [best_bounds[instance_name] + inc for inc in range(2)]
    for bound in bounds_to_insert:
        instance_name_with_bound = f"{instance_name}_{bound}"
        write_plain_asp_file(data, instance_name_with_bound, dir_name = dir_name, mode=mode)
        with open(f"{dir_name}/{instance_name_with_bound}.asp", "a") as f:
            f.write("\n")
            f.write("\n".join([create_rule(f"lb({bound})")] + lines))

In [23]:
import pandas as pd

best_bounds_map: dict[str, int] = {}
df_best_bounds = pd.read_csv("cat_optimums.csv", sep=";")
for id, row in df_best_bounds.iterrows():
    instance = row["instance"]
    bound = row["opt"]
    instance = instance.replace(".txt","")
    best_bounds_map[instance] = int(bound)

best_bounds_map
# df_best_bounds

{'cat_paths_60_100_0000': 16542,
 'cat_paths_60_100_0001': 15748,
 'cat_paths_60_100_0002': 18588,
 'cat_paths_60_100_0003': 17250,
 'cat_paths_60_100_0004': 15364,
 'cat_paths_60_100_0005': 14726,
 'cat_paths_60_100_0006': 14961,
 'cat_paths_60_100_0007': 15329,
 'cat_paths_60_110_0000': 15911,
 'cat_paths_60_110_0001': 14710,
 'cat_paths_60_110_0002': 18537,
 'cat_paths_60_110_0003': 20084,
 'cat_paths_60_110_0004': 14100,
 'cat_paths_60_110_0005': 15505,
 'cat_paths_60_110_0006': 17530,
 'cat_paths_60_110_0007': 15423,
 'cat_paths_60_120_0000': 19685,
 'cat_paths_60_120_0001': 13006,
 'cat_paths_60_120_0002': 14745,
 'cat_paths_60_120_0003': 18110,
 'cat_paths_60_120_0004': 18078,
 'cat_paths_60_120_0005': 19373,
 'cat_paths_60_120_0006': 15460,
 'cat_paths_60_120_0007': 20470,
 'cat_paths_60_130_0000': 18606,
 'cat_paths_60_130_0001': 14138,
 'cat_paths_60_130_0002': 18528,
 'cat_paths_60_130_0003': 17691,
 'cat_paths_60_130_0004': 17507,
 'cat_paths_60_130_0005': 19931,
 'cat_path

In [24]:
best_bounds_map["cat_paths_60_70_0000"]

14730

In [25]:
import os
from pathlib import Path

# instances_dir_amo = f"instances_amosum_{'_'.join(str(e).replace(".",",") for e in P)}"
instances_dir_amo = f"instances_amosum_best_bounds"
# instances_dir_plain = "instances_plain"
instances_dly = "instances_dly"

Path(instances_dir_amo).mkdir(exist_ok=True)
for f in os.listdir(instances_dir_amo):
    os.remove(f"{instances_dir_amo}/{f}")

# for f in os.listdir(instances_dir_plain):
#     os.remove(f"{instances_dir_plain}/{f}")

for f in os.listdir(instances_dly):
    parsed_file = parse_dzn_file(f"{instances_dly}/{f}")
    asp_instance_name = f.replace(".dly","")
    # write_plain_asp_file(parsed_file, asp_instance_name)
    write_amosum_asp_file(parsed_file,  asp_instance_name, instances_dir_amo, best_bounds=best_bounds_map)


content:
['num_items=65;\n', 'num_bids=218;\n', 'max_item=65;\n', 'index_packages=[1,9,17,25,33,41,49,57,65,73,81,89,97,105,113,121,129,137,145,153,161,169,177,185,193,201,209,217,225,233,241,249,257,265,273,281,289,297,305,313,321,329,337,345,353,361,369,377,385,393,401,409,417,425,433,442,451,460,469,478,487,496,505,514,523,532,541,550,559,568,577,586,595,604,613,622,631,640,651,662,673,684,695,706,717,728,739,750,761,772,783,794,805,816,827,838,849,860,871,882,893,904,915,926,937,948,959,970,981,992,999,1006,1013,1020,1027,1034,1041,1048,1055,1062,1069,1076,1083,1090,1097,1104,1111,1118,1125,1132,1139,1146,1153,1160,1167,1174,1181,1188,1195,1202,1209,1216,1223,1230,1237,1244,1251,1258,1265,1272,1279,1286,1293,1300,1307,1314,1321,1328,1335,1342,1349,1356,1363,1370,1377,1385,1393,1401,1409,1417,1425,1433,1441,1449,1457,1465,1473,1481,1489,1497,1505,1513,1521,1529,1537,1545,1553,1561,1569,1577,1585,1593,1601,1609,1617,1625,1633,1641,1649,1657,1665,1673,1681,1689,1697,1705,1713,1721,172